# Feast with a Ray offline store on KubeRay

This notebook documents the Feast 0.61 contributed Ray offline store and Ray batch engine on a KubeRay-managed cluster. Feast uses Ray for parallel file-based offline computation; KubeRay provides the separate `RayCluster` and `RayJob` resources. The Feast Operator does not create or manage the Ray cluster.

```text
FileSource (Parquet) -> Feast Ray offline store / Ray batch engine -> registry and online store
                                      ^
                              KubeRay RayCluster
```

Keep this notebook together with `assets/feast-with-ray-offline-store/`. From a Workbench terminal, download both paths with a sparse checkout:

```bash
git clone --depth 1 --filter=blob:none --sparse https://github.com/alauda/aml-docs.git
git -C aml-docs sparse-checkout set --no-cone \
  /docs/en/train/guides/feast-with-ray-offline-store.ipynb \
  /docs/en/train/guides/assets/feast-with-ray-offline-store/
cd aml-docs/docs/en/train/guides
```

Prerequisites:

- the Alauda Build of KubeRay Operator is installed;
- `rayclusters.ray.io` and `rayjobs.ray.io` are available;
- an approved runtime image contains `feast[ray]`, Ray, CodeFlare/KubeRay client dependencies, and the object-store connector required by the `FileSource`;
- the runtime can access the S3-compatible object store and its credentials are supplied by a Secret or workload identity; and
- `kubectl`, `bash`, and `curl` are available in the notebook environment.


## Feast configuration

Configure the Feast repository with a Ray offline store and Ray batch engine. This direct-address mode connects to the Ray Client endpoint exposed by the head Service:

```yaml
offline_store:
  type: ray
  storage_path: s3://<bucket>/feast-data
  ray_address: ray://<ray-head-service>:10001

batch_engine:
  type: ray.engine
  max_workers: 8
```

For CodeFlare/KubeRay submission, use `use_kuberay: true` and configure `kuberay_conf.cluster_name`, `kuberay_conf.namespace`, and the API authentication fields instead of `ray_address`. Set explicit Ray CPU and memory limits, and keep the Ray version in the image, `RayCluster`, and Feast dependencies aligned.

The Ray offline store supports `FileSource` data and has no direct SQL query interface. It does not write an online store by itself; run Feast materialization with the Ray batch engine after offline data is available. See the [Feast Ray offline-store reference](https://docs.feast.dev/v0.61-branch/reference/offline-stores/ray) and [Ray compute-engine reference](https://docs.feast.dev/v0.61-branch/reference/compute-engine/ray).

In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_RAY_ASSET_DIR:-assets/feast-with-ray-offline-store}"
: "${FEAST_RAY_IMAGE:?Set FEAST_RAY_IMAGE to an approved Feast/Ray runtime image}"
test -f "$ASSET_DIR/raycluster.yaml"
test -f "$ASSET_DIR/rayjob.yaml"
kubectl get crd featurestores.feast.dev
kubectl get crd rayclusters.ray.io
kubectl get crd rayjobs.ray.io


## Deploy the KubeRay smoke cluster

The supplied manifest uses a placeholder image. The image must be mirrored into a registry reachable by the workload cluster and must include Feast and Ray.

In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_RAY_ASSET_DIR:-assets/feast-with-ray-offline-store}"
NAMESPACE="${FEAST_RAY_NAMESPACE:-mlops-demo-e2e}"
sed "s#<approved-registry>/mlops/feast-ray-runtime:<tag>#$FEAST_RAY_IMAGE#g" "$ASSET_DIR/raycluster.yaml" | kubectl -n "$NAMESPACE" apply -f -
for _ in $(seq 1 120); do
  state="$(kubectl -n "$NAMESPACE" get raycluster feast-ray-smoke -o jsonpath='{.status.state}' 2>/dev/null || true)"
  echo "RayCluster state=${state:-Pending}"
  [ "$state" = Running ] && break
  sleep 5
done
[ "${state:-}" = Running ] || { kubectl -n "$NAMESPACE" describe raycluster feast-ray-smoke; exit 1; }
kubectl -n "$NAMESPACE" get raycluster feast-ray-smoke -o wide


## Run the Feast/Ray smoke job

The sample job imports Feast and Ray, connects to the running Ray head, and prints the Ray node count. Replace its entrypoint with a small Parquet `FileSource` and `get_historical_features` retrieval when validating a project-specific offline store.

In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_RAY_ASSET_DIR:-assets/feast-with-ray-offline-store}"
NAMESPACE="${FEAST_RAY_NAMESPACE:-mlops-demo-e2e}"
kubectl -n "$NAMESPACE" apply -f "$ASSET_DIR/rayjob.yaml"
for _ in $(seq 1 120); do
  job_status="$(kubectl -n "$NAMESPACE" get rayjob feast-ray-smoke -o jsonpath='{.status.jobStatus}' 2>/dev/null || true)"
  echo "RayJob status=${job_status:-Pending}"
  [ "$job_status" = SUCCEEDED ] && break
  [ "$job_status" = FAILED ] && { kubectl -n "$NAMESPACE" describe rayjob feast-ray-smoke; exit 1; }
  sleep 5
done
[ "${job_status:-}" = SUCCEEDED ] || { kubectl -n "$NAMESPACE" describe rayjob feast-ray-smoke; exit 1; }
kubectl -n "$NAMESPACE" logs -l ray.io/job-name=feast-ray-smoke --tail=100


## Monitoring and lineage

Monitor the Ray cluster and job with `kubectl get raycluster`, `kubectl get rayjob`, pod resource metrics, and the Ray dashboard. Feast's OpenLineage integration emits registry/apply and materialization events, but Feast 0.61 does not expose a native Spark logical-plan or column-lineage hook. For detailed Spark transformation lineage, add an OpenLineage Spark agent to the Spark runtime and configure `spark.extraListeners` and the OpenLineage transport.

Feast quality monitoring in 0.61 is experimental and Great Expectations-based: profile saved datasets or validate logged feature data with `FeatureStore.validate_logged_features`. It is separate from the Ray cluster health checks. See the [Feature quality monitoring reference](https://docs.feast.dev/v0.61-branch/reference/feature-quality-monitoring), [OpenLineage integration](https://docs.feast.dev/v0.61-branch/reference/openlineage), [Alauda KubeRay overview](../../develop/components/kuberay/intro), and [KubeRay demo](https://github.com/alauda/aml-docs/blob/master/docs/public/kuberay/demo.ipynb).

In [ ]:
%%bash
set -euo pipefail
NAMESPACE="${FEAST_RAY_NAMESPACE:-mlops-demo-e2e}"
kubectl -n "$NAMESPACE" delete rayjob feast-ray-smoke --ignore-not-found
kubectl -n "$NAMESPACE" delete raycluster feast-ray-smoke --ignore-not-found
